# Wan 2.2 TI2V — Heritage Short moving shots

**Setup:** Runtime → GPU (T4). Upload `wan_jobs.json` + `keyframes/` (png/jpg) as dataset or input files.

Challenge-aligned: image-to-video from locked keyframes. Motion/camera prompts only.

After run: download `/kaggle/working/shots/*.mp4` into the repo `outputs/shots/`, then locally `npm run shots && npm run assemble`.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
SHOTS_OUT = WORK / 'shots'
SHOTS_OUT.mkdir(parents=True, exist_ok=True)

# Adjust if you uploaded under a dataset name
CANDIDATES = [
    Path('/kaggle/input'),
    WORK,
]

def find_file(name):
    for root in CANDIDATES:
        if not root.exists():
            continue
        for p in root.rglob(name):
            return p
    return None

jobs_path = find_file('wan_jobs.json')
assert jobs_path, 'Upload wan_jobs.json from outputs/shots/'
jobs = json.loads(jobs_path.read_text())
print('jobs', len(jobs['jobs']), 'from', jobs_path)

In [ ]:
# Install / clone Wan 2.2 per current upstream docs when you run this cell.
# https://github.com/Wan-Video/Wan2.2
# https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B
#
# Keep this cell updated to the repo's latest install commands.
print('Install Wan 2.2 TI2V-5B here, then run the generation loop below.')
print('Tip: draft at lower res first; save each shot as you go (sessions time out).')

In [ ]:
def resolve_keyframe(job):
    for name in (job.get('keyframeJpg'), job.get('keyframePng'), job.get('keyframe')):
        if not name:
            continue
        base = Path(name).name
        p = find_file(base)
        if p:
            return p
    return None

for job in jobs['jobs']:
    out = SHOTS_OUT / f"{job['id']}.mp4"
    if out.exists() and out.stat().st_size > 100_000:
        print('skip existing', out.name)
        continue
    img = resolve_keyframe(job)
    if not img:
        print('MISSING keyframe for', job['id'])
        continue
    prompt = job['motionPrompt']
    print('GENERATE', job['id'], 'from', img.name)
    print(' prompt:', prompt[:140])
    # TODO: call your Wan TI2V entrypoint, e.g.:
    # run_ti2v(image=str(img), prompt=prompt, seed=job.get('seed'), out=str(out))
    #
    # Until the model cell is filled, this notebook documents the exact job contract.
print('Done loop scaffold. Wire run_ti2v() to Wan after install cell succeeds.')